# Football3D — HMR2 on KTH Football II, sequence 2, Camera 1

Attach the private SMPL/HMR2 input and the KTH dataset. Enable Internet and a T4 GPU, then run all cells. The KTH 2D joints define the crop; no tracker JSON is needed.

In [ ]:
from pathlib import Path
import numpy as np

INPUT = Path('/kaggle/input')
position_files = list(INPUT.rglob('positions2d.txt'))
assert position_files, 'Upload positions2d.txt together with the KTH PNG frames'
KTH = position_files[0].parent
camera_dirs = list(INPUT.rglob('Camera 1'))
FRAMES = camera_dirs[0] if camera_dirs else next((p.parent for p in INPUT.rglob('00001.png')), None)
assert FRAMES is not None, 'KTH PNG frames were not found under /kaggle/input'
frame_files = sorted(FRAMES.glob('*.png'))
smpl_files = list(INPUT.rglob('basicmodel_m_lbs_10_207_0_v1.1.0.pkl'))
assert len(frame_files) == 175, f'Expected 175 KTH frames, found {len(frame_files)}'
assert smpl_files, 'Attach the private SMPL dataset containing basicmodel_m_lbs_10_207_0_v1.1.0.pkl'
SMPL_SOURCE = smpl_files[0]
POSITIONS_2D = np.loadtxt(position_files[0]).reshape(-1, 3, 14, 2)[:, 0]
START, END = 1, len(frame_files)
WORK = Path('/kaggle/working/football3d_kth_camera1')
WORK.mkdir(parents=True, exist_ok=True)
OUT = WORK / 'kth_camera1_pose.npz'
print('KTH:', KTH)
print('Frames:', len(frame_files), 'SMPL:', SMPL_SOURCE)

In [ ]:
import shutil, subprocess, sys

REPO = Path('/kaggle/working/4D-Humans')
if not (REPO / 'hmr2' / '__init__.py').is_file():
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '-q', 'https://github.com/shubham-goel/4D-Humans.git', str(REPO)], check=True)
assert (REPO / 'hmr2' / '__init__.py').is_file()
subprocess.run(['pip', 'install', '-q', 'aria2'], check=False)
sys.path.insert(0, str(REPO))
print('HMR2 checkout:', REPO)

In [ ]:
import shutil
from hmr2.configs import CACHE_DIR_4DHUMANS

cache_model = Path(CACHE_DIR_4DHUMANS) / 'data' / 'smpl' / 'SMPL_NEUTRAL.pkl'
cache_model.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(SMPL_SOURCE, cache_model)
print('SMPL copied to:', cache_model)

from hmr2.models import DEFAULT_CHECKPOINT
checkpoint = Path(DEFAULT_CHECKPOINT)
if not checkpoint.exists():
    url = 'https://www.cs.utexas.edu/~pavlakos/4dhumans/hmr2_data.tar.gz'
    cache = checkpoint.parent
    cache.mkdir(parents=True, exist_ok=True)
    subprocess.run(['aria2c', '-x', '8', '-s', '8', '-d', str(cache.parent), '-o', 'hmr2_data.tar.gz', url], check=True)
    import tarfile
    with tarfile.open(cache.parent / 'hmr2_data.tar.gz') as archive: archive.extractall(cache.parent)
assert checkpoint.exists(), f'HMR2 checkpoint missing: {checkpoint}'
print('Checkpoint:', checkpoint)

In [ ]:
# Build one tight detector-style box per frame from Camera 1's annotated 2D joints.
boxes = {}
for frame in range(START, END + 1):
    xy = POSITIONS_2D[frame - 1]
    x1, y1 = xy.min(axis=0); x2, y2 = xy.max(axis=0)
    w, h = x2 - x1, y2 - y1
    boxes[frame] = np.array([x1 - .10*w, y1 - .10*h, x2 + .10*w, y2 + .10*h], dtype=np.float32)
print('Boxes:', len(boxes), 'example:', boxes[1])

In [ ]:
import cv2, torch
from hmr2.models import load_hmr2
from hmr2.datasets.vitdet_dataset import ViTDetDataset, DEFAULT_MEAN, DEFAULT_STD
from hmr2.utils import recursive_to
from hmr2.utils.renderer import Renderer

assert torch.cuda.is_available(), 'Enable a T4 GPU in Kaggle settings'
device = torch.device('cuda')
original_torch_load = torch.load
def trusted_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_torch_load(*args, **kwargs)
torch.load = trusted_torch_load
try: model, model_cfg = load_hmr2(str(checkpoint))
finally: torch.load = original_torch_load
model = model.to(device).eval()
renderer = Renderer(model_cfg, faces=model.smpl.faces)
preview_path = WORK / 'kth_camera1_preview.mp4'
preview = subprocess.Popen(['ffmpeg', '-y', '-loglevel', 'error', '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s', '512x256', '-r', '25', '-i', '-', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(preview_path)], stdin=subprocess.PIPE)
frames_out, joints_out, vertices_out = [], [], []
orientations_out, body_poses_out, betas_out = [], [], []
for index, frame in enumerate(sorted(boxes), 1):
    image = cv2.imread(str(FRAMES / f'{frame:05d}.png'))
    dataset = ViTDetDataset(model_cfg, image, boxes[frame][None])
    batch = recursive_to(next(iter(torch.utils.data.DataLoader(dataset, batch_size=1))), device)
    with torch.inference_mode():
        result = model(batch); params = result['pred_smpl_params']
        male = model.smpl(global_orient=params['global_orient'].float(), body_pose=params['body_pose'].float(), betas=torch.zeros_like(params['betas']).float(), pose2rot=False)
    vertices = male.vertices[0]
    joints = torch.einsum('jk,kv->jv', model.smpl.J_regressor.to(device), vertices)
    input_patch = batch['img'][0].cpu() * (DEFAULT_STD[:, None, None] / 255) + (DEFAULT_MEAN[:, None, None] / 255)
    rendered = renderer(vertices.cpu().numpy(), result['pred_cam_t'][0].cpu().numpy(), batch['img'][0])
    preview.stdin.write(np.clip(255 * np.concatenate([input_patch.permute(1, 2, 0).numpy(), rendered], axis=1), 0, 255).astype(np.uint8).tobytes())
    frames_out.append(frame); joints_out.append(joints.cpu().numpy()); vertices_out.append(vertices.cpu().numpy())
    orientations_out.append(params['global_orient'][0].cpu().numpy()); body_poses_out.append(params['body_pose'][0].cpu().numpy()); betas_out.append(params['betas'][0].cpu().numpy())
    if index == 1 or index % 10 == 0: print(f'Processed {index}/{len(boxes)}')
preview.stdin.close(); assert preview.wait() == 0
np.savez_compressed(OUT, frame=np.asarray(frames_out), joints=np.asarray(joints_out), vertices=np.asarray(vertices_out), global_orient_rotmat=np.asarray(orientations_out), body_pose_rotmat=np.asarray(body_poses_out), betas=np.asarray(betas_out), fps=np.float32(25))
print('Saved:', OUT, 'Preview:', preview_path)

In [ ]:
from IPython.display import FileLink, Video, display
display(Video(str(preview_path), embed=True))
display(FileLink(str(OUT)))
display(FileLink(str(preview_path)))